# RAR 150-cycle digits campaign — Kaggle (resumable, headless)
**Settings:** Internet ON, Accelerator = None (CPU). Add `OPENROUTER_API_KEY` as a Secret.
**Run:** Save Version -> *Save & Run All (Commit)* — runs detached up to ~12h, survives disconnect.
**Resume:** add this notebook's previous Output as an Input dataset; finished seeds skip.


### 1. Clone instrumented repo + install (CPU)

In [ ]:
import os, sys, glob, shutil, subprocess
# Internet must be ON (Notebook settings -> Internet). Accelerator: None (CPU).
REPO_DIR = "/kaggle/working/recursive-autonomy-research"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Tahir-yamin/recursive-autonomy-research.git", REPO_DIR], check=True)
os.chdir(REPO_DIR)
assert "per_cycle_trajectories" in open("run_pilot_experiment.py").read(), "repo not instrumented"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "numpy", "scipy", "scikit-learn", "networkx", "aiohttp", "matplotlib"], check=True)
print("setup OK:", os.getcwd())

### 2. Restore finished seeds from a prior run (resume)

In [ ]:
# RESUME: copy any finished seed files from a prior run (mounted under /kaggle/input)
# into the repo dir, so run_one_seed skips them. Add this notebook's previous
# output as an input dataset to chain commits.
import glob, shutil, os
restored = []
for src in glob.glob("/kaggle/input/**/pilot_seed_*_digits.json", recursive=True):
    if "classical" in src:
        continue
    dst = os.path.basename(src)
    if not os.path.exists(dst):
        shutil.copy2(src, dst); restored.append(dst)
print("restored finished seeds:", sorted(restored) or "none (fresh start)")

### 3. API key from Kaggle Secrets

In [ ]:
import os
# Add OPENROUTER_API_KEY via Add-ons -> Secrets, then attach to this notebook.
from kaggle_secrets import UserSecretsClient
key = UserSecretsClient().get_secret("OPENROUTER_API_KEY")
assert key, "OPENROUTER_API_KEY secret missing"
os.environ["OPENROUTER_API_KEY"] = key
os.environ.pop("RAR_SIM", None)          # force real inference
os.environ["RAR_DATASET"] = "digits"
os.environ["RAR_CYCLES"] = "150"
print("key loaded, real-inference mode, digits, 150 cycles")

### 4. Pre-flight model select

In [ ]:
import sys, asyncio
for m in ("run_pilot_experiment", "run_deep_learning_harness"):
    sys.modules.pop(m, None)
import run_pilot_experiment as rpe
CANDIDATES = ["openai/gpt-oss-20b:free", "meta-llama/llama-3.3-70b-instruct:free",
              "google/gemma-3-12b-it:free", "deepseek/deepseek-chat-v3:free"]
async def preflight():
    for slug in CANDIDATES:
        os.environ["OPENROUTER_MODEL"] = slug
        try:
            r = await rpe.call_llm("Reply with the single word: OK")
        except Exception as e:
            print("x", slug, str(e)[:60]); continue
        if r:
            print("ok ->", slug); return slug
    raise RuntimeError("no free model responded")
chosen = asyncio.get_event_loop().run_until_complete(preflight())
os.environ["OPENROUTER_MODEL"] = chosen

### 5. Run campaign (finished seeds skip; output persists on commit)

In [ ]:
import asyncio, os
DATASET = "digits"
async def run_one_seed(seed):
    out = f"pilot_seed_{seed}_{DATASET}.json"
    if os.path.exists(out):
        print(f"seed {seed}: already done, skip"); return
    print(f"===== SEED {seed} =====")
    os.environ["RAR_OUTPUT_FILE"] = out
    rpe.SEEDS = [seed]; rpe.CYCLES = int(os.environ["RAR_CYCLES"])
    await rpe.execute_campaign()
    print(f"seed {seed}: DONE -> {out}" if os.path.exists(out) else f"seed {seed}: NO FILE")

SEEDS = [42, 7, 13, 23, 88, 99, 101, 107, 113, 127]
loop = asyncio.get_event_loop()
for s in SEEDS:
    loop.run_until_complete(run_one_seed(s))   # finished seeds in /kaggle/working persist on commit
print("loop done")

### 6. CRS / frontier-proximity trajectory

In [ ]:
# ============================================================================
# CORRECTED Cell 26 -- REAL Context-Rot (CRS) trajectory.
# Replaces the broken version that plotted per-SEED redundancy counts as if
# they were cycles. This reads the genuine per-cycle trajectories that the
# INSTRUMENTED run_pilot_experiment.py now logs under
# conditions[*]["per_cycle_trajectories"]. If that field is absent (i.e. the
# campaign was run with the OLD orchestrator), it refuses and tells you to
# re-run -- it does NOT fabricate a trajectory.
#
# Requires: your repo's run_pilot_experiment.py must be the instrumented
# version (it persists per_cycle_trajectories). Re-run the seed cells (13-22)
# AFTER updating the repo, then run this cell.
# ============================================================================
import json, os, glob
import numpy as np
import matplotlib.pyplot as plt

W = 10  # trailing window
COLORS = {"stateless_baseline": "#b41e1e", "vector_rag": "#1c4f8a", "rar_compressed": "#2ca02c"}
LABELS = {"stateless_baseline": "Stateless", "vector_rag": "Vector RAG", "rar_compressed": "RAR (ours)"}

# Prefer per-seed files (each has one trajectory); fall back to merged file.
DATASET = os.environ.get("RAR_DATASET", "digits")
per_seed = sorted(glob.glob(f"pilot_seed_*_{DATASET}.json"))
conds = {}
if per_seed:
    for f in per_seed:
        c = json.load(open(f)).get("data", {}).get("conditions", {})
        for k, v in c.items():
            conds.setdefault(k, {"per_cycle_trajectories": []})
            conds[k]["per_cycle_trajectories"] += v.get("per_cycle_trajectories", [])
else:
    merged = f"pilot_results_{DATASET}.json"
    if os.path.exists(merged):
        conds = json.load(open(merged)).get("data", {}).get("conditions", {})

have = any(conds.get(c, {}).get("per_cycle_trajectories") for c in conds)
if not have:
    raise SystemExit(
        "No per_cycle_trajectories found. Your campaign was run with the OLD "
        "orchestrator (per-seed means only). Update run_pilot_experiment.py to "
        "the instrumented version, re-run the seed cells, then run this cell. "
        "No trajectory will be fabricated.")


def crs_series(trace):
    trace = sorted(trace, key=lambda r: r["cycle"])
    red = np.array([r["redundant"] for r in trace], float)
    acc = np.array([r["val_acc"] for r in trace], float)
    n = len(trace); NR = np.empty(n); DC = np.empty(n)
    for t in range(n):
        lo = max(0, t - W + 1)
        NR[t] = 1.0 - red[lo:t + 1].mean()
        win = acc[lo:t + 1]
        prior = np.array([acc[:max(1, lo + i)].max() for i in range(len(win))])
        DC[t] = np.mean(win >= prior)
    return (NR + DC) / 2.0, np.array([r["density"] for r in trace])


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.4))
for c in ["stateless_baseline", "vector_rag", "rar_compressed"]:
    trs = [t for t in conds.get(c, {}).get("per_cycle_trajectories", []) if t and len(t) > 1]
    if not trs:
        continue
    L = min(len(t) for t in trs)
    crs = np.array([crs_series(t)[0][:L] for t in trs])
    den = np.array([crs_series(t)[1][:L] for t in trs])
    x = np.arange(1, L + 1)
    ax1.plot(x, crs.mean(0), color=COLORS[c], lw=2, label=LABELS[c])
    ax1.fill_between(x, crs.mean(0) - crs.std(0), crs.mean(0) + crs.std(0), color=COLORS[c], alpha=0.15)
    ax2.plot(x, den.mean(0), color=COLORS[c], lw=2, label=LABELS[c])
ax1.axhline(0.70, color="black", ls=":", lw=1, label="success (0.70)")
ax1.set_xlabel("Cycle"); ax1.set_ylabel("CRS"); ax1.set_title("Coherence Retention Score trajectory"); ax1.legend(fontsize=8); ax1.grid(alpha=0.4)
ax2.set_xlabel("Cycle"); ax2.set_ylabel(r"Context density $\delta$"); ax2.set_title("Context density trajectory"); ax2.legend(fontsize=8); ax2.grid(alpha=0.4)
plt.tight_layout(); plt.savefig("fig_rot_trajectory.png", dpi=300, bbox_inches="tight"); plt.show()
print("Saved REAL fig_rot_trajectory.png  (download this + pilot_results_digits.json and send back).")


### 7. List outputs

In [ ]:
# Outputs persist in /kaggle/working automatically on commit. Download the
# pilot_seed_*_digits.json (+ fig_rot_trajectory.png) from the Output tab,
# or add this notebook's output as an input dataset to the next commit to resume.
import glob; print("outputs:", sorted(glob.glob("/kaggle/working/recursive-autonomy-research/pilot_seed_*_digits.json")))